### 现代卷积神经网络

卷积后的张量可以看作是多个不同的通道层层拼接起来，也可以看作是每个像素有一个表征向量。
对于第一个的理解：每一层都在寻找一种特定的视觉模式。比如第一层在找横线，第二层在找耳朵。早期 CNN 研究者大多有计算机视觉背景，他们将卷积看作一系列滤镜。这个视角解释了平移不变性和空间特征提取。
对于第二个的理解：这个向量是这个像素点的表征。它不再是颜色表示（输入的RGB），而是综合了周围信息后，对这个像素点语义的表示。随着 1x1 卷积、 Transformer、注意力机制的流行，这个视角变得极其重要。因为在这些模型里，我们把每个像素看作一个单词/对象实体，通道数就是它的词向量维度。

#### LeNet
第一个卷积神经网络

#### AlexNet
证明深度的有效性（对比LeNet），以及 Relu、Dropout

#### VGG
使用可复用的卷积块构造网络，3 个串联的 3x3 卷积层，在物理视野（感受野）上，等效于 1 个巨大的 7x7 卷积层。但是参数量只有 7*7 的一半
ResNet、Transformer 等等里面全是的 Block 循环。VGG 就说把 AI 模型推向工业流水线的思想起点

#### NiN
每个NiN Block中在经过一次卷积层后，我们把每个位置的表征向量接入了全连接。实现的操作是用1*1卷积。
且整个架构中，nin_block把表征向量拉长，MaxPool把像素层变窄/面积变小，直到最后把384的表征向量压缩到10，然后nn.AdaptiveAvgPool2d((1, 1))把宽度压缩为1。
实际上就是把一个图层从平面逐渐向深度/长度拉，把面积变小而长度变大，使得NN的宽度变小而深度变大，降低“记忆力”增强“理解力”

CNN就是一种残缺的全连接，叫稀疏连接（Sparse Connectivity） 和 权重共享（Weight Sharing），在扫描整个图层的时候用的是同一套参数。

#### 1×1卷积与全连接的区别
1*1卷积是稀疏且共享参数的，摆脱了需要固定大小的限制，保留了空间结构（全连接要把图层展平而卷积不会）。这种方式即降低了参数量/过拟合的风险，还增加了架构的泛化性。

#### GoogLeNet
Inception 的灵魂在于多尺度特征提取，使用不同的感受野/尺度进行表征，最后进行整合。
有一个细节：Inception 的 1x1 卷积的“减负”功能：1*1卷积就是把输入进的长的表征向量变成窄的，然后下一层大卷积核再进行扩长，这一步强制压缩了表征，增强了泛化性。

#### 批量规范化  Batch Normalization
批次规范化可以看作是每层接受输入时的一次数据预处理，即进行了归一化消除绝对值的影响，防止过大使得数据分布被打破，防止前向传播与梯度爆炸，又引入了噪声增强模型泛化能力。
解决/缓解了内部协变量偏移，使得每一层更加独立


#### ResNet
神经网络学习不动函数即 f(x)=x 是非常难的，这导致了每层都必须对上一层的输入进行一次加工，这可能会导致网络越深，而无意义的折腾带来的误差就越大。
所以对每一层加上一个特殊通道 x --> x，这样就可以从拟合 f(x)=x 变为f(x)=0，而拟合残差是非常容易的，这是由于loss里面的正则化项

与此同时，数据在流经网络时，每经过一层都可以选择两条路（走卷积，或者走跨层线）。一个 N 层的 ResNet，有着 $2^{N}$ 种不同的路径。它的冗余度非常大。

在第一个版本的ResNet，跨层连接的x还要经过这一层的激活函数，这导致还存在梯度回传失败的风险，第二版本直接把BatchNorm和激活函数都放入了主路线，跨层连接直接输送x本身，这导致梯度永远存在


#### DenseNet
把加法变成拼接，保留原始信息。使得每一层都能接受到之前每一层的原始加工处理。Growth Rate 增长率比较小，可能是 32 上下，每一层都只增加 32 防止爆炸。
它有很强的抗过拟合能力，原因在于：保留层层信息而防止了深层特征的“独裁”，参数量和计算量足够小使得模型必须学习真正的通用规律，而防止记忆
Dense Block 内部一直拼接，Transition Layer 提炼核心，一种这一张一弛的结构。
最后拼接使得梯度可以直接回传，而不被 + 混淆淹没


#### ResNet 与 DenseNet（DenseNet的弱点）
DenseNet 需要拼接，每次拼接都需要在显存里重开一块巨大的连续的新内存，把前面的特征图全部复制过去。导致 DenseNet 虽然参数小但是显存占用很大。而 ResNet 每次的加法可以原地操作。
同时 DenseNet 每一层都要去读取前面所有层的特征图（非常碎片化的内存地址），然后用 1*1 卷积拼接特征。计算量小，但读写内存的次数（MAC）非常高。导致 GPU 经常处于停工等待数据输入的状态。

这就是ResNet更适合用于工业实践的原因